<div align="center"><span style="font-family: Arial; color:#0000FF"><b>
    <span style="font-size: x-large">Métodos Numéricos II</span>
    <br>
    <span style="font-size: large">Segundo de Grado en Matemáticas - Curso 2022/23</span>
    <br>
    <span style="font-size: medium">Facultad de Ciencias de la Universidad de Málaga</span>
    <br>
    <span style="font-size: small">Dpto. de Análisis Matemático, Estadística e Investigación Operativa, y Matemática Aplicada</span>
    <br>
    <span style="font-size: small">Profs. Manuel J. Castro y Francisco J. Palma (Área Conocimiento de Matemática Aplicada)</span>
    <br>
    <span style="font-size: medium; color:#FF0000">Práctica número 8</span>
    </b></span></div>

In [2]:
from algoritmos import *

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    El objetivo de esta práctica es desarrollar funciones <span style="font-family: Courier">Python</span> para resolver sistemas de ecuaciones lineales mediante los <b>métodos iterativos clásicos de Jacobi, de Gauss-Seidel y de relajación</b>.
    </span></div>

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    Recordamos que, en general, los métodos iterativos para resolver un sistema de ecuaciones lineales compatible y determinado de $n$ ecuaciones con $n$ incógnitas $A\,X=B$ (donde los datos del problema son $A\in\mathcal{M}_n(\mathbb{K})$ inversible y $B\in\mathbb{K}^n$, y la incógnita es $X\in\mathbb{K}^n$), se basan en construir una matriz $C\in\mathcal{M}_n(\mathbb{K})$ y un vector $V\in\mathbb{K}^n$ tales que
\[
A\,X = B \quad \Leftrightarrow \quad X = C\,X +V\,.
\]
    <br>
    A partir de esta formulación equivalente del problema, se construye una sucesión $\{X_k\}_{k\in\mathbb{N}}$ de aproximaciones de la solución de la forma siguiente:
\[
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ X_{k+1} = C\,X_k + V\,, \quad k=0,1,2,\ldots. \end{array} \right.
\]
    <br>
    Es claro que esta construcción de la sucesión permite asegurar que en caso de convergencia lo hará hacia la única solución del problema dado, y sabemos que esa convergencia se produce si y solo si $\rho(C)<1$.
    <br>
    Para la construcción de la matriz $C$ y del vector $V$, se parte de una descomposición de la matriz $A=M-N$, donde $M,N\in\mathcal{M}_n(\mathbb{K})$, con $M$ inversible: se escribe entonces
\[
A\,X = B \quad \Leftrightarrow \quad X = M^{-1}\,N\,X + M^{-1}\,B\,,
\]
con lo que $C=M^{-1}\,N$ y $V=M^{-1}\,B$.
    <br>
    Los métodos iterativos clásicos utilizan la descomposición de la matriz $A$ en la forma
\[
A = D - E - F\,,
\]
donde las matrices $D,E,F\in\mathcal{M}_n(\mathbb{K})$ son, respectivamente, diagonal, estrictamente triangular inferior y estrictamente triangular superior. Hacemos siempre la hipótesis que los elementos diagonales de $A$ son no nulos, lo que asegura que la matriz $D$ siempre es inversible.
    </span></div>

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    El <b>método iterativo de Jacobi</b> se basa en la elección $M=D$ y $N=E+F$, con lo que se tiene que $C=D^{-1}\,(E+F)$ (esta matriz se suele notar mediante $J$) y $V=D^{-1}\,B$. La sucesión generada $\{X_k\}_{k\in\mathbb{N}}$ viene dada por
\[
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ X_{k+1} = D^{-1}\,(E+F)\,X_k + D^{-1}\,B\,, \quad k=0,1,2,\ldots, \end{array} \right.
\]
o equivalentemente
\[
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ D\,X_{k+1} = B + (E+F)\,X_k\,, \quad k=0,1,2,\ldots. \end{array} \right.
\]
    <br>
    Si ponemos $X_k=(x_i^{(k)})_{i=1}^n$, entonces
\[
x_i^{(k+1)} = \frac{1}{a_{i,i}}\,\left( b_i - \sum_{j=1\,,\,j\ne i}^n a_{i,j}\,x_j^{(k)} \right)\,.
\]
    </span></div>

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Ejercicio 1.</b></span> Elaborar un programa de nombre <span style="font-family: Courier">jacobi()</span> que implemente el algoritmo del <b>método iterativo de Jacobi</b>.
    </span></div>

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Observación.</b></span> En todos los programas que siguen, los mismos llevan como parámetros de entrada la matriz $A$, el segundo miembro $B$, el vector $X_0$ con el que iniciar las iteraciones, el número máximo de iteraciones a realizar y la tolerancia del test de parada (que es la norma infinito de la diferencia entre dos iteraciones sucesivas); en el caso del método de relajación, también hay que dar el parámetro de relajación $\omega$.
    </span></div>

In [3]:
def jacobi(A, B, XOLD, itermax, tol):
    m, n = shape(A)
    p, q = shape(B)
    r, s = shape(XOLD)
    if m != n or n != p or q != 1 or n != r or s != 1 or min(abs(diag(A))) < 1e-200:
        return False, 'ERROR jacobi: no se resuelve el sistema.'
    k = 0
    error = 1.
    while k < itermax and error >= tol:
        k = k+1
        XNEW = array(B)
        for i in range(n):
            if i != 0:
                XNEW[i, 0] -= A[i, :i]@XOLD[:i, 0]
            if i != n-1:
                XNEW[i, 0] -= A[i, i+1:]@XOLD[i+1:, 0]
            XNEW[i, 0] = XNEW[i, 0]/A[i, i]
        error = norma_vec(XNEW - XOLD, inf)
        XOLD = array(XNEW)
    print('\nIteración: k = ', k)
    print('Error absoluto: error = ', error)
    if k == itermax and error >= tol:
        return False, 'ERROR jacobi: no se alcanza convergencia.'
    else:
        print('Convergencia numérica alcanzada: jacobi.')
        return True, XNEW

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Ejercicio 2.</b></span> Resolver mediante el método iterativo de Jacobi un sistema lineal $A\,X=B$, cuya matriz de coeficientes $A$ (del tamaño que se quiera) es tridiagonal, con elementos diagonales iguales a 2 y elementos sub-diagonales y supra-diagonales iguales a $-1$ (esta matriz es definida positiva); sabemos que hay convergencia en este caso. Tomamos como segundo miembro $B$ un vector cuyas componentes son las sumas de las respectivas filas de la matriz $A$, lo que nos asegura que la solución exacta del sistema $X$ es el vector con todas las componentes iguales a 1. Tomamos como vector inicial $X_0$ el vector nulo, establecemos un número máximo de iteraciones de 1000 y un valor para la constante de tolerancia de $10^{-5}$.
    </span></div>

In [4]:
# Ejercicio 2
print("Ejercicio 2\n")

n = 5
nMax = 1000
epsilon = 1e-5
A = 2*eye(n) - eye(n, k=1) - eye(n, k=-1)
print('Matriz A = \n', A)

B = reshape(sum(A, axis=1), (n, 1)) # axis = 1 indica las diferentes columnas y axis = 0 indica las filas. Además reshape hace que B tenga 4 filas y 1 columna
print('\nMatriz B = \n', B)

XOLD = zeros((n, 1))
print('\nVector X0 = \n', XOLD)

exito, J = jacobi(A, B, XOLD, nMax, epsilon)
if exito:
    print("\nMatriz X = \n", J)
    print('Comprobacion: ||B - AX||_2 = ', norma_vec(B-A@J, 2))
else:
    print('Error: ', J)

Ejercicio 2

Matriz A = 
 [[ 2. -1.  0.  0.  0.]
 [-1.  2. -1.  0.  0.]
 [ 0. -1.  2. -1.  0.]
 [ 0.  0. -1.  2. -1.]
 [ 0.  0.  0. -1.  2.]]

Matriz B = 
 [[1.]
 [0.]
 [0.]
 [0.]
 [1.]]

Vector X0 = 
 [[0.]
 [0.]
 [0.]
 [0.]
 [0.]]

Iteración: k =  74
Error absoluto: error =  7.945943831355606e-06
Convergencia numérica alcanzada: jacobi.

Matriz X = 
 [[0.99998411]
 [0.99997616]
 [0.99996822]
 [0.99997616]
 [0.99998411]]
Comprobacion: ||B - AX||_2 =  1.9463507911636824e-05


<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    El <b>método iterativo de Gauss-Seidel</b> se basa en la elección $M=D-E$ y $N=F$, con lo que se tiene que $C=(D-E)^{-1}\,F$ (esta matriz se suele notar mediante $\mathcal{L}_1$) y $V=(D-E)^{-1}\,B$. La sucesión generada $\{X_k\}_{k\in\mathbb{N}}$ viene dada por
\[
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ X_{k+1} = (D-E)^{-1}\,F\,X_k + (D-E)^{-1}\,B\,, k=0,1,2,\ldots, \end{array} \right.
\]
o equivalentemente
\[
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ D\,X_{k+1} = B + E\,X_{k+1} + F\,X_k\,, \quad k=0,1,2,\ldots. \end{array} \right.
\]
    <br>
    Si ponemos $X_k=(x_i^{(k)})_{i=1}^n$, entonces
\[
x_i^{(k+1)} = \frac{1}{a_{i,i}}\,\left( b_i - \sum_{j=1}^{i-1} a_{i,j}\,x_j^{(k+1)} - \sum_{j=i+1}^n a_{i,j}\,x_j^{(k)} \right)\,.
\]
    </span></div>

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Ejercicio 3.</b></span> Elaborar un programa de nombre <span style="font-family: Courier">gauss_seidel()</span> que implemente el algoritmo del <b>método iterativo de Gauss-Seidel</b>.
    </span></div>

In [5]:
def gauss_seidel(A, B, XOLD, itermax, tol): # Cambia desde Jacobi X_k a X_k+1
    m, n = shape(A)
    p, q = shape(B)
    r, s = shape(XOLD)
    if m != n or n != p or q != 1 or n != r or s != 1 or min(abs(diag(A))) < 1e-200:
        return False, 'ERROR gauss_seidel: no se resuelve el sistema.'
    k = 0
    error = 1.
    while k < itermax and error >= tol:
        k = k+1
        XNEW = array(B)
        for i in range(n):
            if i != 0: #estos if's los podemos eliminar, pues entra de todos modos
                XNEW[i, 0] -= A[i, :i]@XNEW[:i, 0]
            if i != n-1: #estos if's los podemos eliminar, pues entra de todos modos
                XNEW[i, 0] -= A[i, i+1:]@XOLD[i+1:, 0]
            XNEW[i, 0] = XNEW[i, 0]/A[i, i]
        error = norma_vec(XNEW - XOLD, inf)
        XOLD = array(XNEW)
    print('\nIteración: k = ', k)
    print('Error absoluto: error = ', error)
    if k == itermax and error >= tol:
        return False, 'ERROR gauss_seidel: no se alcanza convergencia.'
    else:
        print('Convergencia numérica alcanzada: gauss-seidel.')
        return True, XNEW

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Ejercicio 4.</b></span> Resolver el mismo sistema que se plantea en el ejercicio 2 mediante el método de Gauss-Seidel</b>.
    </span></div>

In [6]:
# Ejercicio 4
print("Ejercicio 4\n")

n = 5
nMax = 1000
epsilon = 1e-5
A = 2*eye(n) - eye(n, k=1) - eye(n, k=-1)
print('Matriz A = \n', A)

B = reshape(sum(A, axis=1), (n, 1)) # axis = 1 indica las columnas y axis = 0 indica las filas. Además reshape hace que B tenga 4 filas y 1 columna
print('\nMatriz B = \n', B)

XOLD = zeros((n, 1))
print('\nVector X0 = \n', XOLD)

exito, G = gauss_seidel(A, B, XOLD, nMax, epsilon)
if exito:
    print("\nMatriz X = \n", G)
    print('Comprobacion: ||B - AX||_2 = ', norma_vec(B-A@G, 2))
else:
    print('Error: ', G)

Ejercicio 4

Matriz A = 
 [[ 2. -1.  0.  0.  0.]
 [-1.  2. -1.  0.  0.]
 [ 0. -1.  2. -1.  0.]
 [ 0.  0. -1.  2. -1.]
 [ 0.  0.  0. -1.  2.]]

Matriz B = 
 [[1.]
 [0.]
 [0.]
 [0.]
 [1.]]

Vector X0 = 
 [[0.]
 [0.]
 [0.]
 [0.]
 [0.]]

Iteración: k =  38
Error absoluto: error =  8.277024824421275e-06
Convergencia numérica alcanzada: gauss-seidel.

Matriz X = 
 [[0.99998345]
 [0.99997517]
 [0.99997517]
 [0.99998138]
 [0.99999069]]
Comprobacion: ||B - AX||_2 =  1.3608408022659568e-05


<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    Finalmente la familia de <b>métodos iterativos de relajación</b> se basa en la elección $M=\frac{1}{\omega}\,D-E$ y $N=\frac{1-\omega}{\omega}\,D+F$, donde $\omega\in\mathbb{R}-\{0\}$, con lo que se tiene que $C=\left(\frac{1}{\omega}\,D-E\right)^{-1}\,\left(\frac{1-\omega}{\omega}\,D+F\right)$ (esta matriz se suele notar mediante $\mathcal{L}_\omega$) y $V=\left(\frac{1}{\omega}\,D-E\right)^{-1}\,B$. La sucesión generada $\{X_k\}_{k\in\mathbb{N}}$ viene dada por
\[
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ X_{k+1} = \left(\frac{1}{\omega}\,D-E\right)^{-1}\,\left(\frac{1-\omega}{\omega}\,D+F\right)\,X_k + \left(\frac{1}{\omega}\,D-E\right)^{-1}\,B\,, \quad k=0,1,2,\ldots, \end{array} \right.
\]
o equivalentemente
\[
\left\{ \begin{array}{l} X_0 \in \mathbb{K}^n \quad \mbox{dado}\,, \\ \frac{1}{\omega}\,D\,X_{k+1} = B + E\,X_{k+1} + \frac{1-\omega}{\omega}\,D\,X_k + F\,X_k\,, \quad k=0,1,2,\ldots. \end{array} \right.
\]
    <br>
    Si ponemos $X_k=(x_i^{(k)})_{i=1}^n$, entonces
\[
x_i^{(k+1)} = \frac{\omega}{a_{i,i}}\,\left( b_i - \sum_{j=1}^{i-1} a_{i,j}\,x_j^{(k+1)} + \frac{1-\omega}{\omega}\,a_{i,i}\,x_i^{(k)} - \sum_{j=i+1}^n a_{i,j}\,x_j^{(k)} \right)\,.
\]
    </span></div>

In [7]:
def relajacion(A, B, XOLD, omega, itermax, tol):
    (m, n) = shape(A)
    (p, q) = shape(B)
    (r, s) = shape(XOLD)
    if m != n or n != p or q != 1 or n != r or s != 1 or min(abs(diag(A))) < 1e-10:
        return False, 'ERROR relajacion: no se resuelve el sistema.'
    k = 0
    error = 1.
    while k < itermax and error >= tol:
        k = k+1
        XNEW = array(B)
        for i in range(n):
            if i != 0: #estos if's los podemos eliminar, pues entra de todos modos
                XNEW[i, 0] = XNEW[i, 0] - A[i, :i]@XNEW[:i, 0]
            if i != n-1: #estos if's los podemos eliminar, pues entra de todos modos
                XNEW[i, 0] = XNEW[i, 0] - A[i, i+1:]@XOLD[i+1:, 0]
            XNEW[i, 0] += ((1 - omega)/omega)*A[i, i]*XOLD[i, 0]
            XNEW[i, 0] = omega*XNEW[i, 0]/A[i, i]
        error = norma_vec(XNEW - XOLD, inf)
        XOLD = array(XNEW)
    print('\nIteración: k = ', k)
    print('Error: error = ', error)
    if k == itermax and error >= tol:
        return False, 'ERROR relajacion: no se resuelve el sistema.'
    else:
        print('Convergencia numérica alcanzada: relajación.')
        return True, XNEW

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Ejercicio 5.</b></span> Resolver el mismo sistema que se plantea en el ejercicio 2 mediante el método de relajación, tomando diferentes valores del parámetro $\omega$</b>.
    </span></div>

In [8]:
# Ejercicio 5
print("Ejercicio 5\n")

n = 5
nMax = 1000
epsilon = 1e-5
A = 2*eye(n) - eye(n, k=1) - eye(n, k=-1)
print('Matriz A = \n', A)

B = reshape(sum(A, axis=1), (n, 1)) # axis = 1 indica las columnas y axis = 0 indica las filas. Además reshape hace que B tenga 4 filas y 1 columna
print('\nMatriz B = \n', B)

XOLD = zeros((n, 1))
print('\nVector X0 = \n', XOLD)

exito, G = relajacion(A, B, XOLD, 1, nMax, epsilon)
if exito:
    print("\nMatriz X = \n", G)
    print('Comprobacion: ||B - AX||_2 = ', norma_vec(B-A@G, 2))
else:
    print('Error: ', G)

Ejercicio 5

Matriz A = 
 [[ 2. -1.  0.  0.  0.]
 [-1.  2. -1.  0.  0.]
 [ 0. -1.  2. -1.  0.]
 [ 0.  0. -1.  2. -1.]
 [ 0.  0.  0. -1.  2.]]

Matriz B = 
 [[1.]
 [0.]
 [0.]
 [0.]
 [1.]]

Vector X0 = 
 [[0.]
 [0.]
 [0.]
 [0.]
 [0.]]

Iteración: k =  38
Error: error =  8.277024824421275e-06
Convergencia numérica alcanzada: relajación.

Matriz X = 
 [[0.99998345]
 [0.99997517]
 [0.99997517]
 [0.99998138]
 [0.99999069]]
Comprobacion: ||B - AX||_2 =  1.3608408022659568e-05


<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Ejercicio 6.</b></span> Para un caso concreto de matriz tridiagonal definida positiva, calcular el parámetro óptimo de relajación, y realizar diferentes ensayos de dicho método, utilizando el valor óptimo de $\omega$, así como valores inferiores y superiores.
    </span></div>

In [11]:
# Ejercicio 6
print("Ejercicio 6\n")

n = 5
A = 2*eye(n) - eye(n, k=1) - eye(n, k=-1)
print('Matriz A = \n', A)

Dinv= inv(diagflat(diag(A)))

F = -triu(A,k=1)
E = -tril(A,k=-1)
J = Dinv@(E+F)
print("\nMatriz D = \n",Dinv)
print("\nMatriz J = \n",J)
autoval ,autovec = eig(J)
ro = max(abs(autoval))
print("\n",ro)
print(autoval)
w0 = 2 / (1 + sqrt(1-ro**2)) # parámetro óptimo de relajación
print("\n",w0)

Ejercicio 6

Matriz A = 
 [[ 2. -1.  0.  0.  0.]
 [-1.  2. -1.  0.  0.]
 [ 0. -1.  2. -1.  0.]
 [ 0.  0. -1.  2. -1.]
 [ 0.  0.  0. -1.  2.]]

Matriz D = 
 [[0.5 0.  0.  0.  0. ]
 [0.  0.5 0.  0.  0. ]
 [0.  0.  0.5 0.  0. ]
 [0.  0.  0.  0.5 0. ]
 [0.  0.  0.  0.  0.5]]

Matriz J = 
 [[0.  0.5 0.  0.  0. ]
 [0.5 0.  0.5 0.  0. ]
 [0.  0.5 0.  0.5 0. ]
 [0.  0.  0.5 0.  0.5]
 [0.  0.  0.  0.5 0. ]]

 0.86602540378444
[ 8.66025404e-01 -8.66025404e-01 -5.00000000e-01 -1.64912105e-17
  5.00000000e-01]

 1.3333333333333355


<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Ejercicio 7.</b></span> En el caso de matrices tridiagonales y, en general, banda con semianchura de banda $p$, optimizar los programas anteriores de manera que se eviten las operaciones innecesarias.
    </span></div>

<div align="left"><span style="font-family: Arial; color:#000000; font-size: medium">
    <span style="color:#FF0000"><b>Ejercicio 8.</b></span> Modificar el método de Jacobi para que el test de parada sea ||Rk+1-Rk||$<$tol con Rk+1=B-A*Xk+1, en lugar de ||Xk+1-Xk||$<$tol
    </span></div>

In [1]:
def jacobi_mod(A, B, XOLD, itermax, tol):
    m, n = shape(A)
    p, q = shape(B)
    r, s = shape(XOLD)
    if m != n or n != p or q != 1 or n != r or s != 1 or min(abs(diag(A))) < 1e-200:
        return False, 'ERROR jacobi: no se resuelve el sistema.'
    k = 0
    error = 1.
    
    Rold = B-A@XOLD
    
    while k < itermax and error >= tol:
        k = k+1
        XNEW = array(B)
        for i in range(n):
            if i != 0:
                XNEW[i, 0] -= A[i, :i]@XOLD[:i, 0]
            if i != n-1:
                XNEW[i, 0] -= A[i, i+1:]@XOLD[i+1:, 0]
            XNEW[i, 0] = XNEW[i, 0]/A[i, i]
            
            Rnew = B-A@XNEW
            
        error = norma_vec(Rnew-Rold, inf)
        
        Rold = array(Rnew)
        XOLD = array(XNEW)
        
    print('\nIteración: k = ', k)
    print('Error absoluto: error = ', error)
    if k == itermax and error >= tol:
        return False, 'ERROR jacobi: no se alcanza convergencia.'
    else:
        print('Convergencia numérica alcanzada: jacobi.')
        return True, XNEW

In [2]:
# Ejercicio 8
print("Ejercicio 8\n")

n = 5
nMax = 1000
epsilon = 1e-5
A = 2*eye(n) - eye(n, k=1) - eye(n, k=-1)
print('Matriz A = \n', A)

B = reshape(sum(A, axis=1), (n, 1)) # axis = 1 indica las columnas y axis = 0 indica las filas. Además reshape hace que B tenga 4 filas y 1 columna
print('\nMatriz B = \n', B)

XOLD = zeros((n, 1))
print('\nVector X0 = \n', XOLD)

exito, X = jacobi_mod(A, B, XOLD, nMax, epsilon)
if exito:
    print("\nMatriz X = \n", X)
    print('Comprobacion: ||B - AX||_2 = ', norma_vec(B-A@X, 2))
else:
    print('Error: ', X)

Ejercicio 8



NameError: name 'eye' is not defined